In [ ]:
%load_ext autoreload
%autoreload 2

### **Paso 1: Generación de datos**

In [ ]:
# Generación de los datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.PanelCreditSimulator import PanelCreditSimulator
from playground.generate_simple_panel import generate_panel

In [ ]:
# Paso 1: generar los datos (por ahora hacemos uno muy sintético)
panel = generate_panel()

# Caso real
# panel_simulator = PanelCreditSimulator(DATA_CONFIG)
# panel = panel_simulator.simulate_panel()

In [ ]:
# Paso 1.1: análisis exploratorio de los datos
print(panel.head())
print()
print(panel.info())
print()
print(list(panel.columns))

### **Paso 2: Split en train y test**

In [ ]:
# Divisón de IDs en train y test
from splits.split_generator import SplitGenerator

In [ ]:
# Paso 2: generar el split train/test. El split_generator ya está configurado
# para que al train vayan todos los tratados y un porcentaje de los nini, y al
# test vayan todos los controles y el resto de los nini
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=13)
train_ids, test_ids = split_generator.generate()

In [ ]:
last = panel.sort_values('t').groupby('firm_id').last()
status = last[['treated', 'control', 'cohort']].reset_index()
status

In [ ]:
# Paso 2.1: revisar que el split se hizo correctamente
split = split_generator.split

train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == True, f"Firm {id} is not treated"
    assert firm["control"].iloc[0] == False, f"Firm {id} is control"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"
    assert firm["treated"].iloc[0] == False, f"Firm {id} is not treated"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"

### **Paso 3: Conversión de datos a tensores de PyTorch y formamos `Datasets`**

In [ ]:
from connectors.lstm import LSTMConnector

In [ ]:
lstm_connector = LSTMConnector(
    panel=panel,
    split=split_generator.split,
    feature_cols=['y_1', 'y_2']
)

train_dataset, test_dataset = lstm_connector.convert(fit_scaler=False)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))

print(train_dataset[0])
print(test_dataset[0])

In [ ]:
for (*X, y) in train_dataset[:10]:
    temporal = X[0]
    print(temporal.shape)   # Vemos que son secuencias de largo variable

# for (*X, y) in test_dataset[:10]:
#     temporal = X[0]
#     print(temporal.shape)

### **Paso 4: Creamos los DataLoaders**

Hacer esto no es tan directo porque tenemos secuencias de largo variable.
Existen dos alternativas:
1. Usar `padding` y `packing`.
2. Organizar los lotes de tal manera que cada lote tenga secuencias de la misma
longitud.

Por ahora, vamos con la opción 1.

Algunas referencias:
- [Discuss PyTorch - Different length sequences as batch input to LSTM](https://discuss.pytorch.org/t/different-length-sequences-as-batch-input-to-lstm/66119)
- [Discuss PyTorch - Understanding pack_padded_sequence and pad_packed_sequence](https://discuss.pytorch.org/t/understanding-pack-padded-sequence-and-pad-packed-sequence/4099/15)

In [ ]:
# Lo probamos primero con un ejemplo
# Supongamos que nuestro lote es el siguiente
batch = [train_dataset[i][0] for i in range(3)]
for elem in batch:
    print(elem.shape)  # Vemos que son secuencias de largo variable

In [ ]:
from torch.nn.utils.rnn import pad_sequence
batch_padded = pad_sequence(batch, batch_first=True)
for elem in batch_padded:
    print(elem.shape)  # Vemos que ahora todas las secuencias tienen el mismo largo

In [ ]:
import torch

def collate_fn(batch):
    """
    batch: lista de (sequence, cohort, label)
    """
    sequences, cohorts, labels = zip(*batch)

    # Tenemos que devolver los largo originales para que el modelo sepa hasta
    # dónde leer (esto después se le pasa a pack_padded_sequence)
    lengths = torch.tensor([s.shape[0] for s in sequences], dtype=torch.long)

    # pad_sequence apila y rellena con 0s hasta la longitud máxima del batch
    # sequences_padded: (batch_size, T_max, F)
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)

    return (
        sequences_padded,
        lengths,
        torch.stack(cohorts),
        torch.stack(labels),
    )

In [ ]:
from torch.utils.data import DataLoader
from connectors.lstm import PanelSequenceDataset

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=PanelSequenceDataset.collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=PanelSequenceDataset.collate_fn
)

In [ ]:
# Veamos un batch de ejemplo
batch = next(iter(train_loader))

print(f"Secuencias: {batch[0].shape}\n")        # (batch_size, T_max, F)
print(f"Largos originales:\n\t{batch[1]}\n")    # (batch_size,)
print(f"IDs de cohortes: \n\t{batch[2]}\n")     # (batch_size,)
print(f"Labels: \n\t{batch[3]}")                # (batch_size,)

### **Paso 5: Instanciar el modelo**

In [ ]:
from models.lstm_classifier import LSTMClassifier

In [ ]:
model = LSTMClassifier(
    n_features=2,
    lstm_hidden_size=64,
    lstm_num_layers=1,
    n_cohorts=3,
    dropout=0.3
)

In [ ]:
# Probamos que el modelo ande bien con una muestra del dataset
seq, cohort, label = train_dataset[0]

print(seq.shape)
print(cohort.shape)

lengths = torch.tensor([len(seq)], dtype=torch.long)
print(lengths, lengths.shape)

# El modelo espera (batch_size, seq_len, n_features) así que agregamos la
# dimensión de batch
logit = model(seq.unsqueeze(0), lengths, cohort.unsqueeze(0))
print(logit)         # tensor con un valor
print(logit.shape)   # torch.Size([1, 1])

In [ ]:
# Probemos con un batch
batch = next(iter(train_loader))
seqs, lengths, cohorts, labels = batch

logit = model(seqs, lengths, cohorts)
print(logit)         # tensor con un valor por cada muestra del batch
print(logit.shape)   # torch.Size([batch_size, 1])

### **Paso 6: Definir la función de pérdida y el optimizador**

In [ ]:
import torch.nn as nn

n_nini    = len(split_generator.split['train']['NiNi'])
n_treated = len(split_generator.split['train']['T'])
pos_weight = torch.tensor(n_nini * model.n_cohorts / n_treated, dtype=torch.float32)

print(f"Pos weight: {pos_weight}")

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### **Paso 7: Entrenamiento**

In [ ]:
from training.trainer import Trainer

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=loss_fn,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)

In [ ]:
trainer.fit(train_loader, test_loader, n_epochs=10)